# save-results

Saves standardised result tables for one experimental condition (NW topology + params).
Run this notebook once after each batch of simulations completes.

Output directory: `results/summary/{topology}/{net_params}/{exp_id}/`

| File | Contents | Granularity |
|------|----------|-------------|
| `selected_seeds.json` | The 50 seeds used (reproducible) | — |
| `nw_stats_5000step.csv` | gamma, clustering, density per seed × direction × step | 5000-step snapshots |
| `post_count_100step.csv` | Mean posts per direction-relative class per seed × direction | 100-step windows |
| `post_prob_100step.csv` | Mean postProb per direction-relative class per seed × direction | 100-step windows |
| `follower_composition_5000step.csv` | Target hub follower opinion class fractions per seed × direction × step | 5000-step snapshots |
| `unfollow_by_class.csv` | Unfollow counts pre/post manipulation per seed × direction | 2 windows |
| `posting_prob_increase_by_class.csv` | Agents with ↑ postProb, count + rate per seed × direction | 2 windows |

**Class labels** are always direction-relative (`far_opposite` … `far_target`), never raw bin indices.

In [1]:
import os
import re
import glob
import json
import yaml
import hashlib
import numpy as np
import pandas as pd
import networkx as nx
import shutil
import xml.etree.ElementTree as ET

RESULTS_DIR  = "./results"
SUMMARY_ROOT = "./results/summary"
N_AGENTS     = 1000
MAX_SEEDS    = 50
RNG_SEED     = 42   # fixed for reproducibility

GEXF_STEPS   = [0, 5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000]
BINS         = ["bin_0", "bin_1", "bin_2", "bin_3", "bin_4"]
GROUP_ORDER  = ['far_opposite', 'near_opposite', 'neutral', 'near_target', 'far_target']

_POS_MAP = {
     1.0: {0: 'far_opposite', 1: 'near_opposite', 2: 'neutral', 3: 'near_target', 4: 'far_target'},
    -1.0: {0: 'far_target',   1: 'near_target',   2: 'neutral', 3: 'near_opposite', 4: 'far_opposite'},
}

# ---------------------------------------------------------------------------
# GEXF helpers
# ---------------------------------------------------------------------------
def _strip_ns(root):
    for elem in root.iter():
        if '}' in elem.tag:
            elem.tag = elem.tag.split('}', 1)[1]

def _gexf_path(run_dir, step):
    fs = glob.glob(os.path.join(run_dir, 'GEXF', '*', f'step_{step}.gexf'))
    return fs[0] if fs else None

def _gexf_has_target(fpath):
    try:
        tree = ET.parse(fpath); root = tree.getroot(); _strip_ns(root)
        attr_map = {a.get('id'): a.get('title')
                    for a in root.findall(".//attributes[@class='node']/attribute")}
        for node in root.findall('.//node'):
            for av in node.findall('.//attvalue'):
                if attr_map.get(av.get('for'), av.get('for')) == 'target' \
                        and av.get('value', '').lower() == 'true':
                    return True
    except Exception:
        pass
    return False

def parse_gexf_full(fpath):
    """Returns (DiGraph, target_id_or_None, {nid: {opinionclass, postprob}})."""
    try:
        tree = ET.parse(fpath); root = tree.getroot(); _strip_ns(root)
        attr_map = {a.get('id'): a.get('title')
                    for a in root.findall(".//attributes[@class='node']/attribute")}
        G = nx.DiGraph()
        target_id = None
        agents = {}
        for node in root.findall('.//node'):
            nid = node.get('id')
            G.add_node(nid)
            d = {'opinionclass': -1, 'postprob': np.nan}
            for av in node.findall('.//attvalue'):
                t, v = attr_map.get(av.get('for'), av.get('for')), av.get('value', '')
                if t == 'opinionClass':
                    try: d['opinionclass'] = int(v)
                    except: pass
                elif t == 'postProb':
                    try: d['postprob'] = float(v)
                    except: pass
                elif t == 'target' and v.lower() == 'true':
                    target_id = nid
            agents[nid] = d
        for edge in root.findall('.//edge'):
            G.add_edge(edge.get('source'), edge.get('target'))
        return G, target_id, agents
    except Exception:
        return None, None, {}

# ---------------------------------------------------------------------------
# MLE power-law gamma on in-degree (Clauset et al. 2009)
# ---------------------------------------------------------------------------
def _mle_powerlaw(k):
    k = k[k >= 1]
    if len(k) < 5:
        return np.nan
    candidates = np.unique(k)
    best_ks, best_kmin = np.inf, candidates[0]
    for kmin in candidates:
        tail = k[k >= kmin]
        n = len(tail)
        if n < 5: break
        g = 1.0 + n / np.sum(np.log(tail / (kmin - 0.5)))
        xs = np.sort(tail)
        ecdf = np.arange(1, n + 1) / n
        tcdf = 1.0 - (xs / kmin) ** (-(g - 1))
        ks = np.max(np.abs(ecdf - tcdf))
        if ks < best_ks:
            best_ks, best_kmin = ks, kmin
    tail_f = k[k >= best_kmin]
    n_f = len(tail_f)
    if n_f < 5: return np.nan
    return 1.0 + n_f / np.sum(np.log(tail_f / (best_kmin - 0.5)))

# ---------------------------------------------------------------------------
# Config loader (mirrors result-summary.ipynb)
# ---------------------------------------------------------------------------
def load_yaml_manual(fpath):
    config = {}; current_section = None
    try:
        with open(fpath, 'r') as f:
            for line in f:
                raw = line.rstrip(); line = raw.strip()
                if not line or line.startswith('#'): continue
                indent = len(raw) - len(line)
                if ':' in line:
                    key, val = line.split(':', 1)
                    key = key.strip(); val = val.strip().split('#')[0].strip()
                    if val.isdigit(): val = int(val)
                    elif val.replace('.','',1).isdigit(): val = float(val)
                    elif val.lower() == 'true': val = True
                    elif val.lower() == 'false': val = False
                    elif val.startswith('"') and val.endswith('"'): val = val[1:-1]
                    if indent == 0:
                        if not val: current_section = key; config[current_section] = {}
                        else: config[key] = val; current_section = None
                    elif current_section:
                        config[current_section][key] = val
    except: pass
    return config

print('Helpers loaded.')


Helpers loaded.


In [2]:
# ---------------------------------------------------------------------------
# 1. Detect valid seeds (target node found at step 20000)
# ---------------------------------------------------------------------------
def build_valid_seeds(results_dir, check_step=20000):
    seed_dirs = glob.glob(os.path.join(results_dir, 'run_*_dir_*'))
    seeds_found = set()
    for d in seed_dirs:
        m = re.match(r'run_(\d+)_dir_', os.path.basename(d))
        if m: seeds_found.add(int(m.group(1)))

    valid, excluded = [], []
    for seed in sorted(seeds_found):
        gexf_files = glob.glob(
            os.path.join(results_dir, f'run_{seed}_dir_*', 'GEXF', '*', f'step_{check_step}.gexf'))
        if any(_gexf_has_target(f) for f in gexf_files):
            valid.append(seed)
        else:
            excluded.append(seed)
    print(f'Valid seeds  : {len(valid)}  {valid}')
    print(f'Excluded     : {len(excluded)}  {excluded}')
    return valid

ALL_VALID_SEEDS = build_valid_seeds(RESULTS_DIR)

# ---------------------------------------------------------------------------
# 2. Select up to MAX_SEEDS with fixed RNG
# ---------------------------------------------------------------------------
rng = np.random.default_rng(RNG_SEED)
if len(ALL_VALID_SEEDS) <= MAX_SEEDS:
    SELECTED_SEEDS = sorted(ALL_VALID_SEEDS)
    print(f'All {len(SELECTED_SEEDS)} valid seeds used (< {MAX_SEEDS}).')
else:
    SELECTED_SEEDS = sorted(rng.choice(ALL_VALID_SEEDS, MAX_SEEDS, replace=False).tolist())
    print(f'Randomly selected {MAX_SEEDS} seeds from {len(ALL_VALID_SEEDS)}: {SELECTED_SEEDS}')

# ---------------------------------------------------------------------------
# 3. Determine output directory from config.yaml
# ---------------------------------------------------------------------------
cfg_path = './config.yaml'
config   = load_yaml_manual(cfg_path) if os.path.exists(cfg_path) else {}

param_str = json.dumps(config, sort_keys=True, default=str)
exp_id    = hashlib.md5(param_str.encode()).hexdigest()[:8]

topo    = config.get('topology', 'Unknown')
net_p   = config.get('network_params', {})
net_str = '_'.join(f'{k}_{net_p[k]}' for k in sorted(net_p)) if net_p else 'default'

DEST_DIR = os.path.join(SUMMARY_ROOT, topo, net_str, exp_id)
os.makedirs(DEST_DIR, exist_ok=True)
print(f'Output dir: {DEST_DIR}')

# Save selected seeds
shutil.copy(cfg_path, os.path.join(DEST_DIR, 'config.yaml'))
print('config.yaml saved.')

with open(os.path.join(DEST_DIR, 'selected_seeds.json'), 'w') as f:
    json.dump({'rng_seed': RNG_SEED, 'n_valid': len(ALL_VALID_SEEDS),
               'selected': SELECTED_SEEDS}, f, indent=2)
print('selected_seeds.json saved.')


Valid seeds  : 64  [2, 4, 5, 6, 8, 9, 10, 12, 14, 16, 17, 19, 20, 22, 24, 27, 29, 30, 31, 32, 36, 37, 41, 42, 45, 46, 47, 48, 49, 52, 53, 54, 55, 57, 58, 59, 61, 63, 64, 66, 67, 69, 70, 71, 72, 73, 74, 76, 77, 78, 80, 81, 83, 85, 86, 88, 89, 91, 93, 94, 95, 96, 97, 98]
Excluded     : 36  [0, 1, 3, 7, 11, 13, 15, 18, 21, 23, 25, 26, 28, 33, 34, 35, 38, 39, 40, 43, 44, 50, 51, 56, 60, 62, 65, 68, 75, 79, 82, 84, 87, 90, 92, 99]
Randomly selected 50 seeds from 64: [4, 5, 6, 8, 10, 12, 14, 16, 17, 19, 20, 22, 24, 27, 29, 30, 32, 36, 37, 42, 46, 47, 48, 49, 53, 54, 57, 58, 59, 63, 64, 66, 69, 70, 71, 72, 74, 77, 78, 80, 81, 85, 88, 89, 91, 93, 94, 95, 96, 97]
Output dir: ./results/summary/HolmeKim/A_1_m_3_pt_0.3/bc4819ae
config.yaml saved.
selected_seeds.json saved.


In [3]:
# ---------------------------------------------------------------------------
# Item 1: NW stats at 5000-step intervals
# Source: degrees/degree_result_{step}.csv  → gamma, mean_in_degree, density
#         clusterings/clustering_result_{step}.csv → avg_clustering
# ---------------------------------------------------------------------------
nw_records = []

for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir):
            continue

        for step in GEXF_STEPS:
            rec = {'seed': seed, 'target_sign': target_sign, 'step': step,
                   'gamma': np.nan, 'mean_in_degree': np.nan,
                   'density': np.nan, 'avg_clustering': np.nan}

            # degree CSV
            deg_path = os.path.join(run_dir, 'degrees', f'degree_result_{step}.csv')
            if os.path.exists(deg_path):
                try:
                    df_deg = pd.read_csv(deg_path)
                    in_deg = df_deg['inDegree'].values.astype(float)
                    rec['gamma']         = _mle_powerlaw(in_deg)
                    rec['mean_in_degree'] = float(np.mean(in_deg))
                    rec['density']       = float(np.mean(in_deg) / (N_AGENTS - 1))
                except Exception as e:
                    print(f'[seed={seed} dir={target_sign} step={step}] degree error: {e}')

            # clustering CSV (not saved at step 0 by default)
            clu_path = os.path.join(run_dir, 'clusterings', f'clustering_result_{step}.csv')
            if os.path.exists(clu_path):
                try:
                    df_clu = pd.read_csv(clu_path)
                    rec['avg_clustering'] = float(df_clu['clusteringCoefficient'].mean())
                except Exception as e:
                    print(f'[seed={seed} dir={target_sign} step={step}] clustering error: {e}')

            nw_records.append(rec)

df_nw = pd.DataFrame(nw_records)
out_path = os.path.join(DEST_DIR, 'nw_stats_5000step.csv')
df_nw.to_csv(out_path, index=False)
print(f'nw_stats_5000step.csv saved  ({len(df_nw)} rows)')
df_nw.head()

nw_stats_5000step.csv saved  (900 rows)


,seed,target_sign,step,gamma,mean_in_degree,density,avg_clustering
0,4,1.0,0,2.016235,3.000,0.003003,NaN
1,4,1.0,5000,1.769706,3.004,0.003007,0.360039
2,4,1.0,10000,1.784517,2.924,0.002927,0.372164
3,4,1.0,15000,1.821935,2.951,0.002954,0.374153
4,4,1.0,20000,1.793184,2.967,0.002970,0.374511


In [4]:
# ---------------------------------------------------------------------------
# Item 2: Post count per direction-relative class (100-step windows)
# Source: posts/post_result_*.csv
# ---------------------------------------------------------------------------
WINDOW = 100

def load_posts_timeseries(run_dir, target_sign, window=WINDOW):
    """Returns DataFrame with columns [seed, target_sign, step, far_opposite, ..., far_target]."""
    post_dir = os.path.join(run_dir, 'posts')
    files = glob.glob(os.path.join(post_dir, 'post_result_*.csv'))
    if not files: return None
    dfs = []
    for f in files:
        try: dfs.append(pd.read_csv(f))
        except: pass
    if not dfs: return None
    df = pd.concat(dfs).sort_values('step').reset_index(drop=True)

    pmap = _POS_MAP[target_sign]
    # rename bin_i → relative group
    bin_to_group = {f'bin_{i}': pmap[i] for i in range(5)}
    for b, g in bin_to_group.items():
        if b in df.columns:
            df[g] = df.get(g, 0) + df[b]

    df['window_idx'] = df['step'] // window
    agg = df.groupby('window_idx')[GROUP_ORDER].mean()
    agg['step'] = agg.index * window
    return agg.reset_index(drop=True)

post_count_records = []
for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir): continue
        df_ts = load_posts_timeseries(run_dir, target_sign)
        if df_ts is None: continue
        df_ts.insert(0, 'target_sign', target_sign)
        df_ts.insert(0, 'seed', seed)
        post_count_records.append(df_ts)

df_post_count = pd.concat(post_count_records, ignore_index=True)
cols = ['seed', 'target_sign', 'step'] + GROUP_ORDER
df_post_count = df_post_count[cols]
out_path = os.path.join(DEST_DIR, 'post_count_100step.csv')
df_post_count.to_csv(out_path, index=False)
print(f'post_count_100step.csv saved  ({len(df_post_count)} rows)')
df_post_count.head()

post_count_100step.csv saved  (40100 rows)


,seed,target_sign,step,far_opposite,near_opposite,neutral,near_target,far_target
0,4,1.0,0,0.70,1.78,3.08,2.58,1.40
1,4,1.0,100,0.67,1.64,2.81,2.02,1.10
2,4,1.0,200,0.79,1.28,2.45,2.20,1.06
3,4,1.0,300,0.82,1.26,2.18,1.35,0.92
4,4,1.0,400,0.91,1.42,2.00,1.72,0.73


In [5]:
# ---------------------------------------------------------------------------
# Item 3: Post probability per direction-relative class (100-step windows)
# Source: metrics/result_*.csv  columns postProbMean_0 … postProbMean_4
# NOTE: requires simulation rebuilt after Java changes (postProbMean columns added)
# ---------------------------------------------------------------------------
PP_COLS = [f'postProbMean_{i}' for i in range(5)]

def load_postprob_timeseries(run_dir, target_sign, window=WINDOW):
    metrics_dir = os.path.join(run_dir, 'metrics')
    files = glob.glob(os.path.join(metrics_dir, 'result_*.csv'))
    if not files: return None
    dfs = []
    for f in files:
        try: dfs.append(pd.read_csv(f))
        except: pass
    if not dfs: return None
    df = pd.concat(dfs).sort_values('step').reset_index(drop=True)

    missing = [c for c in PP_COLS if c not in df.columns]
    if missing:
        print(f'  [WARN] columns not found: {missing} — skipping (re-run simulation first)')
        return None

    pmap = _POS_MAP[target_sign]
    for i in range(5):
        grp = pmap[i]
        df[grp] = df.get(grp, 0.0) + df[f'postProbMean_{i}']

    # Each group might aggregate two bins (for groups that don't overlap with another bin,
    # there will be no double-counting since _POS_MAP is a bijection)
    df['window_idx'] = df['step'] // window
    agg = df.groupby('window_idx')[GROUP_ORDER].mean()
    agg['step'] = agg.index * window
    return agg.reset_index(drop=True)

post_prob_records = []
for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir): continue
        df_ts = load_postprob_timeseries(run_dir, target_sign)
        if df_ts is None: continue
        df_ts.insert(0, 'target_sign', target_sign)
        df_ts.insert(0, 'seed', seed)
        post_prob_records.append(df_ts)

if post_prob_records:
    df_post_prob = pd.concat(post_prob_records, ignore_index=True)
    cols = ['seed', 'target_sign', 'step'] + GROUP_ORDER
    df_post_prob = df_post_prob[cols]
    out_path = os.path.join(DEST_DIR, 'post_prob_100step.csv')
    df_post_prob.to_csv(out_path, index=False)
    print(f'post_prob_100step.csv saved  ({len(df_post_prob)} rows)')
    display(df_post_prob.head())
else:
    print('post_prob_100step.csv NOT saved — no postProbMean columns found.')
    print('Re-run simulations with the updated Java build first.')

post_prob_100step.csv saved  (40100 rows)


,seed,target_sign,step,far_opposite,near_opposite,neutral,near_target,far_target
0,4,1.0,0,0.085987,0.085193,0.091607,0.101379,0.089770
1,4,1.0,100,0.072097,0.070677,0.081942,0.090157,0.088168
2,4,1.0,200,0.067571,0.063244,0.071187,0.078835,0.085678
3,4,1.0,300,0.072905,0.057260,0.061801,0.071318,0.084597
4,4,1.0,400,0.076347,0.057191,0.059947,0.068764,0.074037


In [6]:
# ---------------------------------------------------------------------------
# Item 4: Target hub follower opinion composition (5000-step snapshots)
# Target node identified from step-20000 GEXF, then tracked back/forward.
# Follower opinion class is read from the snapshot at each respective step.
# ---------------------------------------------------------------------------
fol_records = []
print('Item 4: collecting follower compositions ...')

for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir): continue

        # Identify target node at step 20000
        fp_20k = _gexf_path(run_dir, 20000)
        if not fp_20k: continue
        _, tid, _ = parse_gexf_full(fp_20k)
        if tid is None: continue

        for step in GEXF_STEPS:
            fp = _gexf_path(run_dir, step)
            if not fp: continue
            G, _, agents = parse_gexf_full(fp)
            if G is None or tid not in G: continue

            followers = list(G.predecessors(tid))
            counts = {g: 0 for g in GROUP_ORDER}
            total  = 0
            for f in followers:
                oc = agents.get(f, {}).get('opinionclass', -1)
                if oc not in range(5): continue
                grp = _POS_MAP[target_sign].get(oc, 'unknown')
                if grp in counts:
                    counts[grp] += 1
                    total += 1

            row = {'seed': seed, 'target_sign': target_sign, 'step': step,
                   'total_followers': total}
            for g in GROUP_ORDER:
                row[g] = counts[g] / total if total > 0 else np.nan
            fol_records.append(row)

df_fol = pd.DataFrame(fol_records)
cols = ['seed', 'target_sign', 'step', 'total_followers'] + GROUP_ORDER
df_fol = df_fol[cols]
out_path = os.path.join(DEST_DIR, 'follower_composition_5000step.csv')
df_fol.to_csv(out_path, index=False)
print(f'follower_composition_5000step.csv saved  ({len(df_fol)} rows)')
df_fol.head()

Item 4: collecting follower compositions ...


follower_composition_5000step.csv saved  (900 rows)


,seed,target_sign,step,total_followers,far_opposite,near_opposite,neutral,near_target,far_target
0,4,1.0,0,102,0.147059,0.186275,0.235294,0.186275,0.245098
1,4,1.0,5000,106,0.188679,0.245283,0.264151,0.254717,0.047170
2,4,1.0,10000,113,0.106195,0.230088,0.442478,0.194690,0.026549
3,4,1.0,15000,106,0.075472,0.264151,0.433962,0.207547,0.018868
4,4,1.0,20000,106,0.075472,0.273585,0.433962,0.207547,0.009434


In [7]:
# ---------------------------------------------------------------------------
# Item 5: Unfollow counts by class — pre (0→20000) and post (20000→40000)
# Unfollowers classified by opinionClass at step 20000 (last pre-intervention snapshot).
# ---------------------------------------------------------------------------
unfollow_records = []
print('Item 5: collecting unfollow counts ...')

for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir): continue

        fp_20k = _gexf_path(run_dir, 20000)
        if not fp_20k: continue
        G_20k, tid, agents_20k = parse_gexf_full(fp_20k)
        if tid is None or G_20k is None: continue

        def _cls(nid):
            oc = agents_20k.get(nid, {}).get('opinionclass', -1)
            return _POS_MAP[target_sign].get(oc, 'unknown') if oc in range(5) else 'unknown'

        fol_20k = set(G_20k.predecessors(tid))

        fol_0 = set()
        fp_0 = _gexf_path(run_dir, 0)
        if fp_0:
            G_0, _, _ = parse_gexf_full(fp_0)
            if G_0 is not None and tid in G_0:
                fol_0 = set(G_0.predecessors(tid))

        fol_40k = set()
        fp_40k = _gexf_path(run_dir, 40000)
        if fp_40k:
            G_40k, _, _ = parse_gexf_full(fp_40k)
            if G_40k is not None and tid in G_40k:
                fol_40k = set(G_40k.predecessors(tid))

        for window, unfol in [('pre_0_20000', fol_0 - fol_20k),
                               ('post_20000_40000', fol_20k - fol_40k)]:
            counts = {g: 0 for g in GROUP_ORDER}
            for nid in unfol:
                grp = _cls(nid)
                if grp in counts:
                    counts[grp] += 1
            row = {'seed': seed, 'target_sign': target_sign, 'window': window}
            row.update(counts)
            unfollow_records.append(row)

df_unfollow = pd.DataFrame(unfollow_records)
cols = ['seed', 'target_sign', 'window'] + GROUP_ORDER
df_unfollow = df_unfollow[cols]
out_path = os.path.join(DEST_DIR, 'unfollow_by_class.csv')
df_unfollow.to_csv(out_path, index=False)
print(f'unfollow_by_class.csv saved  ({len(df_unfollow)} rows)')
df_unfollow.head()

Item 5: collecting unfollow counts ...


unfollow_by_class.csv saved  (200 rows)


,seed,target_sign,window,far_opposite,near_opposite,neutral,near_target,far_target
0,4,1.0,pre_0_20000,9,5,3,12,23
1,4,1.0,post_20000_40000,8,29,46,7,0
2,4,-1.0,pre_0_20000,23,12,3,5,9
3,4,-1.0,post_20000_40000,1,13,15,2,0
4,5,1.0,pre_0_20000,33,12,2,5,16


In [8]:
# ---------------------------------------------------------------------------
# Item 6: Agents with increased posting probability by class
# pre  window: steps 15000 & 20000 (snapshots just before/at manipulation)
# post window: steps 35000 & 40000 (snapshots near end of simulation)
# opinionClass assigned from step 15000 (last pre-manipulation class snapshot).
# Target node identified from step 20000.
# ---------------------------------------------------------------------------
_A6_PRE  = [15000, 20000]
_A6_POST = [35000, 40000]

def _load_agents(run_dir, step):
    fp = _gexf_path(run_dir, step)
    if not fp: return {}
    _, _, agents = parse_gexf_full(fp)
    return agents

pp_increase_records = []
print('Item 6: counting agents with increased postProb ...')

for seed in SELECTED_SEEDS:
    for target_sign in [1.0, -1.0]:
        run_dir = os.path.join(RESULTS_DIR, f'run_{seed}_dir_{target_sign}')
        if not os.path.isdir(run_dir): continue

        # Identify target from step 20000
        fp_20k = _gexf_path(run_dir, 20000)
        if not fp_20k: continue
        _, tid, _ = parse_gexf_full(fp_20k)
        if tid is None: continue

        # opinionClass reference: step 15000 (pre-manipulation)
        agents_ref = _load_agents(run_dir, 15000)
        if not agents_ref: continue

        pre_snaps  = [_load_agents(run_dir, s) for s in _A6_PRE]
        post_snaps = [_load_agents(run_dir, s) for s in _A6_POST]
        pre_snaps  = [d for d in pre_snaps  if d]
        post_snaps = [d for d in post_snaps if d]
        if not pre_snaps or not post_snaps: continue

        # per-class counters
        n_increased = {g: 0 for g in GROUP_ORDER}
        n_total     = {g: 0 for g in GROUP_ORDER}

        for nid, ref in agents_ref.items():
            if nid == tid: continue  # skip target
            oc = ref.get('opinionclass', -1)
            if oc not in range(5): continue
            grp = _POS_MAP[target_sign].get(oc, 'unknown')
            if grp not in GROUP_ORDER: continue

            pre_pp  = [sn[nid]['postprob'] for sn in pre_snaps
                       if nid in sn and not np.isnan(sn[nid]['postprob'])]
            post_pp = [sn[nid]['postprob'] for sn in post_snaps
                       if nid in sn and not np.isnan(sn[nid]['postprob'])]
            if not pre_pp or not post_pp: continue

            n_total[grp] += 1
            if np.mean(post_pp) > np.mean(pre_pp):
                n_increased[grp] += 1

        for grp in GROUP_ORDER:
            total = n_total[grp]
            pp_increase_records.append({
                'seed': seed, 'target_sign': target_sign,
                'relative_group': grp,
                'n_increased': n_increased[grp],
                'n_total':     total,
                'rate':        n_increased[grp] / total if total > 0 else np.nan,
            })

df_pp_inc = pd.DataFrame(pp_increase_records)
out_path = os.path.join(DEST_DIR, 'posting_prob_increase_by_class.csv')
df_pp_inc.to_csv(out_path, index=False)
print(f'posting_prob_increase_by_class.csv saved  ({len(df_pp_inc)} rows)')
df_pp_inc.head(10)

Item 6: counting agents with increased postProb ...


posting_prob_increase_by_class.csv saved  (500 rows)


,seed,target_sign,relative_group,n_increased,n_total,rate
0,4,1.0,far_opposite,11,164,0.067073
1,4,1.0,near_opposite,33,177,0.186441
2,4,1.0,neutral,43,233,0.184549
3,4,1.0,near_target,45,247,0.182186
4,4,1.0,far_target,57,178,0.320225
5,4,-1.0,far_opposite,45,178,0.252809
6,4,-1.0,near_opposite,36,247,0.145749
7,4,-1.0,neutral,43,233,0.184549
8,4,-1.0,near_target,40,177,0.225989
9,4,-1.0,far_target,11,164,0.067073


In [9]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
saved = [
    'selected_seeds.json',
    'nw_stats_5000step.csv',
    'post_count_100step.csv',
    'post_prob_100step.csv',
    'follower_composition_5000step.csv',
    'unfollow_by_class.csv',
    'posting_prob_increase_by_class.csv',
]
print(f'\nAll outputs in: {DEST_DIR}\n')
for fname in saved:
    fpath = os.path.join(DEST_DIR, fname)
    status = '✓' if os.path.exists(fpath) else '✗ MISSING'
    size = f'{os.path.getsize(fpath)/1024:.1f} KB' if os.path.exists(fpath) else ''
    print(f'  {status}  {fname}  {size}')


All outputs in: ./results/summary/HolmeKim/A_1_m_3_pt_0.3/bc4819ae

  ✓  selected_seeds.json  0.4 KB
  ✓  nw_stats_5000step.csv  63.2 KB
  ✓  post_count_100step.csv  1478.0 KB
  ✓  post_prob_100step.csv  3132.3 KB
  ✓  follower_composition_5000step.csv  87.9 KB
  ✓  unfollow_by_class.csv  6.7 KB
  ✓  posting_prob_increase_by_class.csv  22.0 KB
